# İstanbul İBB Wi-Fi Yeni Kullanıcı Verisi: Mevsimsel Değişim Analizi

Bu çalışma, İstanbul Büyükşehir Belediyesi (İBB) Wi-Fi noktalarına bağlanan yeni kullanıcı verilerini inceleyerek, turist yoğunluğunun yaşandığı yaz ayları ile kış ayları arasındaki kullanıcı dağılımını (yerli/yabancı) oransal olarak karşılaştırmayı amaçlamaktadır.

##  İnceleme Amacı ve Hipotez
İstanbul'da yaz aylarında artan turist sayısının, İBB Wi-Fi hizmetini kullanan yabancı abone oranını kış aylarına kıyasla doğrudan artıracağı varsayılmaktadır. Analizin temel amacı bu varsayımı gerçek verilerle test etmektir.
* **Test Edilecek Önerme:** *"Yabancı abonelerin oranı yaz aylarında (Haziran–Ağustos), kış aylarından (Aralık–Şubat) yüksektir."*

##  Veri Kaynağı ve Özellikleri
Analizde kullanılan veri seti, İstanbul'daki açık internet noktalarına bağlanan kullanıcıların anonimleştirilmiş kayıtlarını içermektedir.
* **Kaynak:** [İBB Açık Veri Portalı](https://data.ibb.gov.tr/)
* **Veri Seti:** Günlük İBB Wi-Fi Yeni Kullanıcı Verisi (2015 - 2021)
* **Boyut:** 73.164 satır, 7 sütun
* **İçerik:** İşlem ID, kayıt tarihi (`SUBSCRIPTION_DATE`), abonelik yapılan ilçe, kullanıcı uyruğu (`SUBSCRIBER_DOMESTIC_FOREIGN` - Yerli/Yabancı), koordinat bilgileri (Enlem/Boylam) ve o lokasyondaki abone sayısı.

##  Sistem Gereksinimleri ve Kullanılan Teknolojiler
Çalışmanın ödev/kısıtlama kuralları gereği, harici görselleştirme (Matplotlib/Seaborn) kütüphaneleri kullanılmadan yalnızca temel veri bilimi yapıları tercih edilmiştir. Bu not defterini çalıştırabilmek için ortamınızda aşağıdaki bileşenlerin yüklü olması gerekmektedir:
* **Ortam:** Python 3.7+ (Jupyter Notebook veya Google Colab)
* **Kütüphaneler:**
  * `pandas` (Veri yükleme, temizleme, gruplama ve manipülasyon)
  * `numpy` (Sayısal hesaplamalar ve mantıksal operatörler)
* **Veri Dosyası:** `Veri_Seti.csv` adlı veri setinin, bu `.ipynb` dosyası ile aynı çalışma dizininde (klasörde) bulunması zorunludur.

## Uygulama Metodolojisi
Analiz süreci sırasıyla şu adımlardan oluşmaktadır:
1. **Veri Yükleme ve Keşif:** Veri setinin çalışma ortamına alınarak boyut, veri tipleri ve eksik değer kontrollerinin yapılması.
2. **Veri Temizleme:** Analiz hedefiyle örtüşmeyen hatalı kayıtların (örn. uyruk bilgisi "Bilinmiyor" olan satırlar) filtrelenerek temizlenmesi.
3. **Özellik Mühendisliği (Feature Engineering):** Tarih sütunundan ay bilgisinin çekilmesi ve yalnızca analiz hedefi olan "Yaz" (6, 7, 8. aylar) ile "Kış" (12, 1, 2. aylar) mevsimlerinin etiketlenmesi.
4. **Matematiksel Analiz:** Mevsimlere göre yerli ve yabancı abone sayılarının toplanıp, yabancı abone yüzdelerinin hesaplanarak önermenin doğrulanması veya çürütülmesi.


## Adım 1: Kütüphaneleri yükle ve veriyi oku

In [ ]:
import pandas as pd
import numpy as np

# Tarihleri okuma sırasında tarih tipine dönüştür
df = pd.read_csv("Veri_Seti.csv", parse_dates=["SUBSCRIPTION_DATE"])

print("Satır ve sütun sayısı:", df.shape)
df.head()

Satır ve sütun sayısı: (73164, 7)


,_id,SUBSCRIPTION_DATE,SUBSCRIPTION_COUNTY,SUBSCRIBER_DOMESTIC_FOREIGN,LONGITUDE,LATITUDE,NUMBER_OF_SUBSCRIBER
0,1,2020-12-27,KADIKÖY,Yabancı,29.023250,40.992700,1
1,2,2021-04-16,ÜMRANİYE,Yerli,29.105516,41.017006,2
2,3,2021-06-25,KADIKÖY,Yerli,29.088630,40.954044,2
3,4,2019-01-23,SARIYER,Yerli,29.090822,41.243239,2
4,5,2021-03-01,KARTAL,Yerli,29.146650,40.901280,1


## Adım 2:  Veri yapısı ve kalite kontrolleri
Eksik değerler, tekrarlar ve analizde kullanılacak kategoriler kontrol edilir.

In [ ]:

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 73164 entries, 0 to 73163
Data columns (total 7 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   _id                          73164 non-null  int64         
 1   SUBSCRIPTION_DATE            73164 non-null  datetime64[us]
 2   SUBSCRIPTION_COUNTY          73164 non-null  str           
 3   SUBSCRIBER_DOMESTIC_FOREIGN  73164 non-null  str           
 4   LONGITUDE                    73164 non-null  float64       
 5   LATITUDE                     73164 non-null  float64       
 6   NUMBER_OF_SUBSCRIBER         73164 non-null  int64         
dtypes: datetime64[us](1), float64(2), int64(2), str(2)
memory usage: 3.9 MB


In [ ]:
print("Tarih aralığı:", df["SUBSCRIPTION_DATE"].min().date(), "–", df["SUBSCRIPTION_DATE"].max().date())
print("\nEksik değer sayıları:")
print(df.isna().sum())
print("\nTamamen tekrar eden satır sayısı:", df.duplicated().sum())
print("\nAbone türü kayıt sayıları:")
print(df["SUBSCRIBER_DOMESTIC_FOREIGN"].value_counts(dropna=False))
print("\nAbone sayısı özeti:")
df["NUMBER_OF_SUBSCRIBER"].describe()

Tarih aralığı: 2015-11-05 – 2021-07-20

Eksik değer sayıları:
_id                            0
SUBSCRIPTION_DATE              0
SUBSCRIPTION_COUNTY            0
SUBSCRIBER_DOMESTIC_FOREIGN    0
LONGITUDE                      0
LATITUDE                       0
NUMBER_OF_SUBSCRIBER           0
dtype: int64

Tamamen tekrar eden satır sayısı: 0

Abone türü kayıt sayıları:
SUBSCRIBER_DOMESTIC_FOREIGN
Yerli         60765
Yabancı       12357
Bilinmiyor       42
Name: count, dtype: int64

Abone sayısı özeti:


count    73164.000000
mean         3.598272
std          6.874500
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max        330.000000
Name: NUMBER_OF_SUBSCRIBER, dtype: float64

In [ ]:

hatali_koordinat = (df["LONGITUDE"] < 27) | (df["LONGITUDE"] > 30)
print("Boylamı 27–30 aralığı dışında kalan satır sayısı:", int(hatali_koordinat.sum()))

Boylamı 27–30 aralığı dışında kalan satır sayısı: 47


## Adım 3: Analiz verisini hazırlama
“Bilinmiyor” kategorisi yerli/yabancı oranı hesabına dahil edilmez. Aralık ayı, izleyen yılın kış sezonuna atanır; böylece örneğin Aralık 2019–Şubat 2020 dönemi 2020 kışı olur.


In [ ]:
analiz = df.loc[
    df["SUBSCRIBER_DOMESTIC_FOREIGN"].isin(["Yerli", "Yabancı"]),
    ["SUBSCRIPTION_DATE", "SUBSCRIBER_DOMESTIC_FOREIGN", "NUMBER_OF_SUBSCRIBER"]
].copy()

analiz["ay"] = analiz["SUBSCRIPTION_DATE"].dt.month
analiz["yil"] = analiz["SUBSCRIPTION_DATE"].dt.year
analiz["mevsim"] = analiz["ay"].map({12: "Kış", 1: "Kış", 2: "Kış", 6: "Yaz", 7: "Yaz", 8: "Yaz"})
analiz["mevsim_yili"] = np.where(analiz["ay"].eq(12), analiz["yil"] + 1, analiz["yil"])

mevsim_verisi = analiz.dropna(subset=["mevsim"]).copy()
print("Yerli/yabancı olarak bilinen kayıt sayısı:", len(analiz))
print("Mevsim karşılaştırmasına giren satır sayısı:", len(mevsim_verisi))
print("Mevsim dışı aylarda kalan satır sayısı:", len(analiz) - len(mevsim_verisi))

Yerli/yabancı olarak bilinen kayıt sayısı: 73122
Mevsim karşılaştırmasına giren satır sayısı: 40906
Mevsim dışı aylarda kalan satır sayısı: 32216


## Adım 4: Mevsim toplamları ve yabancı abone oranı
Oran, yabancı abone sayısının yerli ve yabancı abone toplamına bölünmesiyle hesaplanır. NUMBER_OF_SUBSCRIBER değerleri toplandığı için sonuçlar satır oranı değil, abone sayısı ağırlıklı orandır.

In [ ]:
mevsim_toplam = mevsim_verisi.pivot_table(
    index="mevsim",
    columns="SUBSCRIBER_DOMESTIC_FOREIGN",
    values="NUMBER_OF_SUBSCRIBER",
    aggfunc="sum",
    fill_value=0
)
mevsim_toplam["Toplam abone"] = mevsim_toplam["Yerli"] + mevsim_toplam["Yabancı"]
mevsim_toplam["Yabancı oranı (%)"] = mevsim_toplam["Yabancı"] / mevsim_toplam["Toplam abone"] * 100
mevsim_toplam.round(1)

SUBSCRIBER_DOMESTIC_FOREIGN,Yabancı,Yerli,Toplam abone,Yabancı oranı (%)
mevsim,,,,
Kış,10562,37343,47905,22.0
Yaz,15132,73892,89024,17.0


## Adım 5: Yıl bazında oran ve veri kapsamı
Yıl bazında tablo, toplu oranın hangi yıllardan ve ne ölçüde gözlemden etkilendiğini gösterir. Tarih sayısı, o sezon için CSV’de en az bir kayıt bulunan farklı günlerin sayısıdır; tam veri kapsamını tek başına garanti etmez.

In [ ]:
yil_mevsim = mevsim_verisi.pivot_table(
    index="mevsim_yili",
    columns=["mevsim", "SUBSCRIBER_DOMESTIC_FOREIGN"],
    values="NUMBER_OF_SUBSCRIBER",
    aggfunc="sum",
    fill_value=0
)

# Eksik kategori sütunları varsa sıfırla
yil_mevsim = yil_mevsim.reindex(
    columns=pd.MultiIndex.from_product([["Kış", "Yaz"], ["Yabancı", "Yerli"]]),
    fill_value=0
)

for mevsim in ["Kış", "Yaz"]:
    toplam = yil_mevsim[(mevsim, "Yabancı")] + yil_mevsim[(mevsim, "Yerli")]
    yil_mevsim[(mevsim, "Yabancı oranı (%)")] = np.where(
        toplam > 0,
        yil_mevsim[(mevsim, "Yabancı")] / toplam * 100,
        np.nan
    )

kapsam = mevsim_verisi.groupby(["mevsim_yili", "mevsim"]).agg(
    kayit_satiri=("NUMBER_OF_SUBSCRIBER", "size"),
    veri_bulunan_gun=("SUBSCRIPTION_DATE", "nunique")
)

oran_sutunlari = [(m, "Yabancı oranı (%)") for m in ["Kış", "Yaz"]]
yil_oranlari = yil_mevsim[oran_sutunlari].copy()
yil_oranlari.columns = ["Kış oranı (%)", "Yaz oranı (%)"]

# Kapsam bilgisini satır-sütun yapısına çevirip oranlarla birleştir
kapsam_tablo = kapsam.unstack("mevsim")
kapsam_tablo.columns = [f"{mevsim} {olcu}" for olcu, mevsim in kapsam_tablo.columns]
yil_ozeti = yil_oranlari.join(kapsam_tablo)
yil_ozeti.round(1)

,Kış oranı (%),Yaz oranı (%),Kış kayit_satiri,Yaz kayit_satiri,Kış veri_bulunan_gun,Yaz veri_bulunan_gun
mevsim_yili,,,,,,
2016,0.0,0.0,84,41,63,33
2017,0.0,0.0,35,35,33,32
2018,25.1,10.7,886,1728,90,92
2019,9.1,6.6,1696,2329,90,91
2020,2.9,4.1,1404,112,91,62
2021,25.6,19.9,15420,17136,90,50


## Adım 6: Sonuç
Önermenin yönü, yaz ve kış oranlarının karşılaştırılmasıyla değerlendirilir. Toplu oran veri hacmine göre ağırlıklıdır; yıl bazındaki oranlar ise her yılın görünümünü ayrı ayrı verir.

In [ ]:
kis_orani = mevsim_toplam.loc["Kış", "Yabancı oranı (%)"]
yaz_orani = mevsim_toplam.loc["Yaz", "Yabancı oranı (%)"]
fark = yaz_orani - kis_orani

print(f"Toplu kış yabancı oranı: %{kis_orani:.1f}")
print(f"Toplu yaz yabancı oranı: %{yaz_orani:.1f}")
print(f"Fark (yaz - kış): {fark:.1f} yüzde puan")

if yaz_orani > kis_orani:
    print("Toplu veriye göre önerme destekleniyor.")
else:
    print("Toplu veriye göre önerme desteklenmiyor.")

gecerli_yillar = yil_ozeti.dropna(subset=["Kış oranı (%)", "Yaz oranı (%)"])
if not gecerli_yillar.empty:
    fark_yil = gecerli_yillar["Yaz oranı (%)"] - gecerli_yillar["Kış oranı (%)"]
    print(f"\nHer iki mevsimde veri bulunan sezon yılı sayısı: {len(gecerli_yillar)}")
    print(f"Bu yıllardaki yaz-kış farklarının ortalaması: {fark_yil.mean():.1f} yüzde puan")
    print(f"Yaz oranının kış oranından yüksek olduğu yıl sayısı: {int((fark_yil > 0).sum())} / {len(fark_yil)}")

Toplu kış yabancı oranı: %22.0
Toplu yaz yabancı oranı: %17.0
Fark (yaz - kış): -5.1 yüzde puan
Toplu veriye göre önerme desteklenmiyor.

Her iki mevsimde veri bulunan sezon yılı sayısı: 6
Bu yıllardaki yaz-kış farklarının ortalaması: -3.6 yüzde puan
Yaz oranının kış oranından yüksek olduğu yıl sayısı: 1 / 6


## Hücre 1: Çubuk grafik fonksiyonu

In [1]:

def cubuk(seri, genislik=40):
    en_buyuk = seri.max()                                # en uzun çubuğu referans al
    for etiket, deger in seri.items():
        uzunluk = int(deger / en_buyuk * genislik)       # değeri çubuk uzunluğuna çevir
        print(f"{str(etiket):<12} {'█' * uzunluk} {deger:.1f}")

## Hücre 2: Yaz ve kış yabancı oranı

In [ ]:
print("Yabancı abone oranı (%) - Mevsime göre")
cubuk(tablo["yabanci_orani"])

## Hücre 3: Yıl ve mevsime göre yabancı oranı

In [ ]:

t3 = mv.pivot_table(index=["yil", "mevsim"],
                    columns="SUBSCRIBER_DOMESTIC_FOREIGN",
                    values="NUMBER_OF_SUBSCRIBER",
                    aggfunc="sum")
t3["yabanci_orani"] = t3["Yabancı"] / (t3["Yerli"] + t3["Yabancı"]) * 100


t3.index = [f"{y} {m}" for y, m in t3.index]

print("Yabancı abone oranı (%) - Yıl ve mevsim")
cubuk(t3["yabanci_orani"])

## Hücre 4: Aylara göre yabancı oranı (12 ay)

In [ ]:

aylik = df.pivot_table(index="ay",
                       columns="SUBSCRIBER_DOMESTIC_FOREIGN",
                       values="NUMBER_OF_SUBSCRIBER",
                       aggfunc="sum")
aylik["yabanci_orani"] = aylik["Yabancı"] / (aylik["Yerli"] + aylik["Yabancı"]) * 100

print("Yabancı abone oranı (%) - Aylara göre")
cubuk(aylik["yabanci_orani"])

## Hücre 5: En çok yabancı abone olan 10 ilçe

In [ ]:

ilce = df.pivot_table(index="SUBSCRIPTION_COUNTY",
                      columns="SUBSCRIBER_DOMESTIC_FOREIGN",
                      values="NUMBER_OF_SUBSCRIBER",
                      aggfunc="sum")

print("Yabancı abone sayısı - İlk 10 ilçe")
cubuk(ilce["Yabancı"].sort_values(ascending=False).head(10))

## Hücre 6: Yıllara göre toplam abone

In [ ]:

yillik = df.groupby("yil")["NUMBER_OF_SUBSCRIBER"].sum()

print("Toplam yeni abone - Yıllara göre")
cubuk(yillik)

##  Sonuç ve Değerlendirme Raporu

İBB Açık Veri Portalı'ndan alınan Wi-Fi kullanıcı verileri üzerinde gerçekleştirilen analizler sonucunda, başlangıçta kurulan hipotez test edilmiş ve veri setinden aşağıdaki temel bulgular elde edilmiştir:

### 1. Temel Oransal Bulgular
* **Kış Sezonu Yabancı Abone Oranı:** %22.0
* **Yaz Sezonu Yabancı Abone Oranı:** %17.0

### 2. Hipotez Testi ve "Seyrelme" Etkisi
Elde edilen sonuçlara göre; yabancı abonelerin oranının yaz aylarında artacağı yönündeki başlangıç önermesi **ÇÜRÜTÜLMÜŞTÜR.**

Bunun temel nedeni istatistiksel bir "seyrelme" etkisidir. Yaz aylarında turizme bağlı olarak sisteme dahil olan yabancı abone sayısında mutlak bir artış yaşansa da, yazlık mekanlar ve açık hava alanlarının kullanımındaki artışla birlikte **yerli kullanıcıların sisteme katılımında devasa bir sıçrama** görülmüştür. Yerli abone sayısındaki bu orantısız büyüme, yabancı abonelerin toplam pasta içindeki yüzdelik payını kış aylarına kıyasla aşağı çekmiştir.

### 3. Yıllık Trend Analizi
Durumun tek bir yıla özgü istisnai bir anomali olup olmadığını anlamak adına yapılan yıl bazlı incelemede; analize dahil edilen **6 yılın yalnızca 1'inde** yaz aylarındaki yabancı abone oranının kış aylarını aşabildiği tespit edilmiştir. Bu durum, %17'ye %22'lik oranın rastlantısal olmadığını ve genel bir kullanıcı trendi olduğunu kanıtlamaktadır.

### 4. Analitik Kısıtlar ve Veri Güvenilirliği (Uyarı)
Bu çıkarımlar yapılırken veri setinin doğal kısıtları göz önünde bulundurulmalıdır. Veri setinin kapsadığı tarih aralıklarında (2015-2021) her mevsimin veri uzunluğu ve gün sayısı tamamen eşit veya dengeli dağılmış değildir. Mevsimlerin veri kapsamındaki bu eşitsizlikler nedeniyle, yüzdelik oranlar ve mutlak sayılar üzerinden yapılan bu çıkarımların **temkinli bir şekilde yorumlanması** gerekmektedir.
